# DHIS2 Water Facility Setup

This notebook sets up the Water Facility Tracked Entity in DHIS2, designed to sync with Sunbird RC.

## Prerequisites
- DHIS2 instance running at http://localhost:9090
- Admin credentials (default: admin/district)

## Setup Flow
1. Create Option Sets
2. Create Tracked Entity Attributes
3. Create Tracked Entity Type
4. Create Tracker Program
5. Import Organisation Units

## 0. Setup and Authentication

In [1]:
import requests
import json
import pandas as pd
from IPython.display import display, JSON, Markdown

# Configuration
BASE_URL = "http://localhost:9090/api"
AUTH = ("admin", "district")

HEADERS = {
    "Content-Type": "application/json",
    "Accept": "application/json"
}

# Helper function for API calls
def dhis2_post(endpoint, data):
    response = requests.post(
        f"{BASE_URL}/{endpoint}",
        auth=AUTH,
        headers=HEADERS,
        json=data
    )
    return response

def dhis2_get(endpoint):
    response = requests.get(
        f"{BASE_URL}/{endpoint}",
        auth=AUTH,
        headers=HEADERS
    )
    return response

print(f"DHIS2 API: {BASE_URL}")

DHIS2 API: http://localhost:9090/api


## 1. Check DHIS2 Health

In [2]:
response = dhis2_get("system/info")

if response.status_code == 200:
    info = response.json()
    print(f"DHIS2 Version: {info.get('version')}")
    print(f"Database: {info.get('databaseInfo', {}).get('name')}")
    print(f"Server Date: {info.get('serverDate')}")
else:
    print(f"Error: {response.status_code}")
    print(response.text)

DHIS2 Version: 2.40.11
Database: dhis2
Server Date: 2026-04-27T06:43:43.108


## 1.1 Reset/Cleanup (Optional)

Run this cell to remove all Water Facility metadata and start fresh. 
**Skip this cell if you want to keep existing data.**

In [3]:
# CLEANUP: Remove all Water Facility related metadata
# WARNING: This will delete all data! Only run if you want to start fresh.

RESET_ALL = False  # Change to True to enable cleanup

if RESET_ALL:
    print("Starting cleanup...")
    
    # 1. Delete Program first (depends on tracked entity type)
    response = dhis2_get("programs?filter=code:eq:WF_REGISTRY&fields=id")
    if response.status_code == 200:
        data = response.json()
        for prog in data.get("programs", []):
            del_response = requests.delete(f"{BASE_URL}/programs/{prog['id']}", auth=AUTH, headers=HEADERS)
            if del_response.status_code in [200, 204]:
                print(f"✓ Deleted program: {prog['id']}")
            else:
                print(f"✗ Failed to delete program: {del_response.text}")
    
    # 2. Delete Tracked Entity Type
    response = dhis2_get("trackedEntityTypes?filter=code:eq:WATER_FACILITY&fields=id")
    if response.status_code == 200:
        data = response.json()
        for tet in data.get("trackedEntityTypes", []):
            del_response = requests.delete(f"{BASE_URL}/trackedEntityTypes/{tet['id']}", auth=AUTH, headers=HEADERS)
            if del_response.status_code in [200, 204]:
                print(f"✓ Deleted tracked entity type: {tet['id']}")
            else:
                print(f"✗ Failed to delete TET: {del_response.text}")
    
    # 3. Delete Tracked Entity Attributes
    attr_codes = ["SUNBIRD_OSID", "WF_ID", "SYNC_STATUS_ATTR", "GEO_CODE", "COUNTY", "DISTRICT", 
                  "COMMUNITY", "WATER_POINT_TYPE_ATTR", "EXTRACTION_TYPE_ATTR", "PUMP_TYPE_ATTR",
                  "NUM_TAPS", "HAS_DEPTH_INFO", "DEPTH_METRES", "INSTALLER", "OWNER", "FUNDER", "PHOTO_URL"]
    response = dhis2_get(f"trackedEntityAttributes?filter=code:in:[{','.join(attr_codes)}]&fields=id,code&paging=false")
    if response.status_code == 200:
        data = response.json()
        for attr in data.get("trackedEntityAttributes", []):
            del_response = requests.delete(f"{BASE_URL}/trackedEntityAttributes/{attr['id']}", auth=AUTH, headers=HEADERS)
            if del_response.status_code in [200, 204]:
                print(f"✓ Deleted attribute: {attr['code']}")
            else:
                print(f"✗ Failed to delete attr {attr['code']}: {del_response.status_code}")
    
    # 4. Delete Option Sets and their Options
    os_codes = ["WATER_POINT_TYPE", "EXTRACTION_TYPE", "PUMP_TYPE", "INSTALLER_TYPE", "OWNER_TYPE", "SYNC_STATUS"]
    response = dhis2_get(f"optionSets?filter=code:in:[{','.join(os_codes)}]&fields=id,code,options[id]&paging=false")
    if response.status_code == 200:
        data = response.json()
        for os in data.get("optionSets", []):
            # Delete options first
            for opt in os.get("options", []):
                requests.delete(f"{BASE_URL}/options/{opt['id']}", auth=AUTH, headers=HEADERS)
            # Then delete option set
            del_response = requests.delete(f"{BASE_URL}/optionSets/{os['id']}", auth=AUTH, headers=HEADERS)
            if del_response.status_code in [200, 204]:
                print(f"✓ Deleted option set: {os['code']}")
            else:
                print(f"✗ Failed to delete OS {os['code']}: {del_response.status_code}")
    
    # 5. Delete Organisation Units (optional - be careful!)
    # Uncomment below to also delete org units
    # for code in reversed(list(created_org_units.keys())):
    #     requests.delete(f"{BASE_URL}/organisationUnits/{created_org_units[code]}", auth=AUTH, headers=HEADERS)
    
    print("\nCleanup complete! You can now re-run the setup cells.")
else:
    print("Cleanup skipped. Set RESET_ALL = True to enable.")

Cleanup skipped. Set RESET_ALL = True to enable.


## 2. Create Option Sets

Create option sets for dropdown fields.

In [4]:
# Define all option sets with their options
OPTION_SETS = [
    {
        "name": "Water Point Type",
        "code": "WATER_POINT_TYPE",
        "valueType": "TEXT",
        "options": [
            {"name": "Protected dug well", "code": "PDW"},
            {"name": "Unprotected dug well", "code": "UDW"},
            {"name": "Tube well or borehole", "code": "TWB"},
            {"name": "Protected spring", "code": "PS"},
            {"name": "Unprotected spring", "code": "US"},
            {"name": "Piped water into dwelling/plot/yard", "code": "PWD"},
            {"name": "Public tap/standpipe", "code": "PTS"},
            {"name": "Unequipped borehole", "code": "UEB"},
            {"name": "Rainwater (harvesting)", "code": "RWH"},
            {"name": "Sand/Sub-surface dam", "code": "SSD"},
            {"name": "Other", "code": "OTH"}
        ]
    },
    {
        "name": "Extraction Type",
        "code": "EXTRACTION_TYPE",
        "valueType": "TEXT",
        "options": [
            {"name": "Manual", "code": "MANUAL"},
            {"name": "Electrical", "code": "ELECTRICAL"},
            {"name": "Solar", "code": "SOLAR"},
            {"name": "Other", "code": "OTHER"}
        ]
    },
    {
        "name": "Pump Type",
        "code": "PUMP_TYPE",
        "valueType": "TEXT",
        "options": [
            {"name": "Afridev", "code": "AFRIDEV"},
            {"name": "Consallen", "code": "CONSALLEN"},
            {"name": "India Mark", "code": "INDIA_MARK"},
            {"name": "Kardia", "code": "KARDIA"},
            {"name": "Rope pump", "code": "ROPE_PUMP"},
            {"name": "Vergnet", "code": "VERGNET"},
            {"name": "Other", "code": "OTHER"}
        ]
    },
    {
        "name": "Installer Type",
        "code": "INSTALLER_TYPE",
        "valueType": "TEXT",
        "options": [
            {"name": "Government", "code": "GOVERNMENT"},
            {"name": "NGO", "code": "NGO"},
            {"name": "Private", "code": "PRIVATE"},
            {"name": "Other", "code": "OTHER"}
        ]
    },
    {
        "name": "Owner Type",
        "code": "OWNER_TYPE",
        "valueType": "TEXT",
        "options": [
            {"name": "Community", "code": "COMMUNITY"},
            {"name": "Private Individual", "code": "PRIVATE_INDIVIDUAL"},
            {"name": "School", "code": "SCHOOL"},
            {"name": "NGO", "code": "NGO"},
            {"name": "Health Facility", "code": "HEALTH_FACILITY"},
            {"name": "Other institution", "code": "OTHER_INSTITUTION"},
            {"name": "CBO", "code": "CBO"},
            {"name": "Private", "code": "PRIVATE"},
            {"name": "Unknown", "code": "UNKNOWN"},
            {"name": "Other", "code": "OTHER"}
        ]
    },
    {
        "name": "Sync Status",
        "code": "SYNC_STATUS",
        "valueType": "TEXT",
        "options": [
            {"name": "Pending", "code": "PENDING"},
            {"name": "Synced", "code": "SYNCED"},
            {"name": "Failed", "code": "FAILED"}
        ]
    }
]

print(f"Defined {len(OPTION_SETS)} option sets")
for os in OPTION_SETS:
    print(f"  - {os['name']}: {len(os['options'])} options")

Defined 6 option sets
  - Water Point Type: 11 options
  - Extraction Type: 4 options
  - Pump Type: 7 options
  - Installer Type: 4 options
  - Owner Type: 10 options
  - Sync Status: 3 options


In [5]:
# Check for existing option sets (in case notebook was run before)
existing_codes = ["WATER_POINT_TYPE", "EXTRACTION_TYPE", "PUMP_TYPE", "INSTALLER_TYPE", "OWNER_TYPE", "SYNC_STATUS"]
response = dhis2_get(f"optionSets?filter=code:in:[{','.join(existing_codes)}]&fields=id,name,code")

created_option_sets = {}
if response.status_code == 200:
    data = response.json()
    existing = data.get("optionSets", [])
    if existing:
        print(f"Found {len(existing)} existing option sets (will reuse):")
        for os in existing:
            created_option_sets[os["code"]] = os["id"]
            print(f"  ✓ {os['name']} (ID: {os['id']})")
    else:
        print("No existing option sets found. Will create new ones.")

Found 6 existing option sets (will reuse):
  ✓ Extraction Type (ID: NJCnF7oPvoU)
  ✓ Installer Type (ID: GzkafEbBlmX)
  ✓ Owner Type (ID: eQwgurWONMf)
  ✓ Pump Type (ID: QuFuSrfJoLB)
  ✓ Sync Status (ID: PrhvpRQgZFS)
  ✓ Water Point Type (ID: cogjWo5Iekk)


In [6]:
# Create option sets using metadata endpoint (skip if already exist)
# In DHIS2, options must reference their parent optionSet

if len(created_option_sets) == len(OPTION_SETS):
    print(f"All {len(created_option_sets)} option sets already exist. Skipping creation.")
else:
    for option_set in OPTION_SETS:
        # Skip if already exists
        if option_set["code"] in created_option_sets:
            continue
        
        os_code = option_set["code"]
        
        # Step 1: Create the option set first (without options)
        os_payload = {
            "name": option_set["name"],
            "code": os_code,
            "valueType": option_set["valueType"]
        }
        
        metadata = {"optionSets": [os_payload]}
        response = dhis2_post("metadata", metadata)
        
        if response.status_code not in [200, 201] or response.json().get("status") != "OK":
            print(f"✗ Failed to create option set {os_code}: {response.text}")
            continue
        
        # Get the created option set ID
        get_response = dhis2_get(f"optionSets?filter=code:eq:{os_code}&fields=id")
        if get_response.status_code != 200:
            continue
            
        os_data = get_response.json()
        if not os_data.get("optionSets"):
            continue
            
        os_id = os_data["optionSets"][0]["id"]
        created_option_sets[os_code] = os_id
        
        # Step 2: Create options with reference to the option set
        options_payload = []
        for i, opt in enumerate(option_set["options"]):
            opt_code = f"{os_code}_{opt['code']}"
            options_payload.append({
                "name": opt["name"],
                "code": opt_code,
                "sortOrder": i + 1,
                "optionSet": {"id": os_id}
            })
        
        metadata = {"options": options_payload}
        response = dhis2_post("metadata", metadata)
        
        if response.status_code in [200, 201] and response.json().get("status") == "OK":
            print(f"✓ Created: {option_set['name']} with {len(options_payload)} options (ID: {os_id})")
        else:
            print(f"✗ Failed to create options for {os_code}: {response.text}")

print(f"\nTotal option sets ready: {len(created_option_sets)}")

All 6 option sets already exist. Skipping creation.

Total option sets ready: 6


## 3. Create Tracked Entity Attributes

In [7]:
# Define tracked entity attributes
# Note: optionSet will be added where applicable using the IDs from created_option_sets

ATTRIBUTES = [
    # Sync fields (filled by adapter)
    {
        "name": "Sunbird OSID",
        "shortName": "osid",
        "code": "SUNBIRD_OSID",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "unique": True,
        "searchable": True
    },
    {
        "name": "Water Facility ID",
        "shortName": "wfId",
        "code": "WF_ID",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "unique": True,
        "searchable": True
    },
    {
        "name": "Sync Status",
        "shortName": "syncStatus",
        "code": "SYNC_STATUS_ATTR",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "optionSetCode": "SYNC_STATUS"
    },
    # Location fields
    {
        "name": "Geo Code",
        "shortName": "geoCode",
        "code": "GEO_CODE",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "unique": True,
        "searchable": True
    },
    {
        "name": "County",
        "shortName": "county",
        "code": "COUNTY",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "searchable": True
    },
    {
        "name": "District",
        "shortName": "district",
        "code": "DISTRICT",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "searchable": True
    },
    {
        "name": "Community",
        "shortName": "community",
        "code": "COMMUNITY",
        "valueType": "TEXT",
        "aggregationType": "NONE"
    },
    # Facility details
    {
        "name": "Water Point Type",
        "shortName": "waterPointType",
        "code": "WATER_POINT_TYPE_ATTR",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "optionSetCode": "WATER_POINT_TYPE"
    },
    {
        "name": "Extraction Type",
        "shortName": "extractionType",
        "code": "EXTRACTION_TYPE_ATTR",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "optionSetCode": "EXTRACTION_TYPE"
    },
    {
        "name": "Pump Type",
        "shortName": "pumpType",
        "code": "PUMP_TYPE_ATTR",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "optionSetCode": "PUMP_TYPE"
    },
    {
        "name": "Number of Taps",
        "shortName": "numTaps",
        "code": "NUM_TAPS",
        "valueType": "INTEGER",
        "aggregationType": "SUM"
    },
    {
        "name": "Has Depth Info",
        "shortName": "hasDepthInfo",
        "code": "HAS_DEPTH_INFO",
        "valueType": "BOOLEAN",
        "aggregationType": "NONE"
    },
    {
        "name": "Depth (metres)",
        "shortName": "depthMetres",
        "code": "DEPTH_METRES",
        "valueType": "NUMBER",
        "aggregationType": "AVERAGE"
    },
    # Ownership & Installation
    {
        "name": "Installer",
        "shortName": "installer",
        "code": "INSTALLER",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "optionSetCode": "INSTALLER_TYPE"
    },
    {
        "name": "Owner",
        "shortName": "owner",
        "code": "OWNER",
        "valueType": "TEXT",
        "aggregationType": "NONE",
        "optionSetCode": "OWNER_TYPE"
    },
    {
        "name": "Funder",
        "shortName": "funder",
        "code": "FUNDER",
        "valueType": "TEXT",
        "aggregationType": "NONE"
    },
    {
        "name": "Photo URL",
        "shortName": "photoUrl",
        "code": "PHOTO_URL",
        "valueType": "URL",
        "aggregationType": "NONE"
    }
]

print(f"Defined {len(ATTRIBUTES)} tracked entity attributes")

Defined 17 tracked entity attributes


In [8]:
# Check for existing tracked entity attributes
attr_codes = [a["code"] for a in ATTRIBUTES]
response = dhis2_get(f"trackedEntityAttributes?filter=code:in:[{','.join(attr_codes)}]&fields=id,name,code&paging=false")

created_attributes = {}
if response.status_code == 200:
    data = response.json()
    existing = data.get("trackedEntityAttributes", [])
    if existing:
        print(f"Found {len(existing)} existing attributes (will reuse):")
        for attr in existing:
            # Find searchable info from ATTRIBUTES definition
            attr_def = next((a for a in ATTRIBUTES if a["code"] == attr["code"]), {})
            created_attributes[attr["code"]] = {
                "id": attr["id"],
                "name": attr["name"],
                "searchable": attr_def.get("searchable", False)
            }
            print(f"  ✓ {attr['name']}")
    else:
        print("No existing attributes found. Will create new ones.")

Found 17 existing attributes (will reuse):
  ✓ Community
  ✓ County
  ✓ Depth (metres)
  ✓ District
  ✓ Extraction Type
  ✓ Funder
  ✓ Geo Code
  ✓ Has Depth Info
  ✓ Installer
  ✓ Number of Taps
  ✓ Owner
  ✓ Photo URL
  ✓ Pump Type
  ✓ Sunbird OSID
  ✓ Sync Status
  ✓ Water Facility ID
  ✓ Water Point Type


In [9]:
# Create tracked entity attributes (skip if already exist)

if len(created_attributes) == len(ATTRIBUTES):
    print(f"All {len(created_attributes)} attributes already exist. Skipping creation.")
else:
    for attr in ATTRIBUTES:
        # Skip if already exists
        if attr["code"] in created_attributes:
            continue
            
        # Build attribute payload
        payload = {
            "name": attr["name"],
            "shortName": attr["shortName"],
            "code": attr["code"],
            "valueType": attr["valueType"],
            "aggregationType": attr["aggregationType"]
        }
        
        # Add optional fields
        if attr.get("unique"):
            payload["unique"] = True
        
        # Link to option set if specified
        if attr.get("optionSetCode") and attr["optionSetCode"] in created_option_sets:
            payload["optionSet"] = {"id": created_option_sets[attr["optionSetCode"]]}
        
        # Create via metadata endpoint
        metadata = {"trackedEntityAttributes": [payload]}
        response = dhis2_post("metadata", metadata)
        
        if response.status_code in [200, 201]:
            result = response.json()
            if result.get("status") == "OK":
                # Get the created attribute ID
                get_response = dhis2_get(f"trackedEntityAttributes?filter=code:eq:{attr['code']}&fields=id,name,code")
                if get_response.status_code == 200:
                    attr_data = get_response.json()
                    if attr_data.get("trackedEntityAttributes"):
                        attr_id = attr_data["trackedEntityAttributes"][0]["id"]
                        created_attributes[attr["code"]] = {
                            "id": attr_id,
                            "name": attr["name"],
                            "searchable": attr.get("searchable", False)
                        }
                        print(f"✓ Created: {attr['name']} (ID: {attr_id})")
            else:
                print(f"✗ Failed: {attr['name']} - {result}")
        else:
            print(f"✗ Error: {attr['name']} - {response.status_code}")
            print(response.text)

print(f"\nTotal attributes ready: {len(created_attributes)}")

All 17 attributes already exist. Skipping creation.

Total attributes ready: 17


## 4. Create Tracked Entity Type

In [10]:
# Check for existing tracked entity type and create if needed
response = dhis2_get("trackedEntityTypes?filter=code:eq:WATER_FACILITY&fields=id,name,code")
te_type_id = None

if response.status_code == 200:
    data = response.json()
    if data.get("trackedEntityTypes"):
        te_type_id = data["trackedEntityTypes"][0]["id"]
        print(f"✓ Tracked Entity Type already exists: Water Facility (ID: {te_type_id})")
    else:
        # Build tracked entity type attributes list
        te_type_attributes = []
        for i, (code, attr_info) in enumerate(created_attributes.items()):
            te_type_attributes.append({
                "trackedEntityAttribute": {"id": attr_info["id"]},
                "displayInList": code in ["SUNBIRD_OSID", "WF_ID", "SYNC_STATUS_ATTR", "GEO_CODE", "COUNTY", "DISTRICT", "COMMUNITY", "WATER_POINT_TYPE_ATTR"],
                "searchable": attr_info.get("searchable", False),
                "sortOrder": i + 1
            })

        # Create tracked entity type
        tracked_entity_type = {
            "name": "Water Facility",
            "shortName": "WaterFacility",
            "code": "WATER_FACILITY",
            "description": "Water facilities with Sunbird RC integration",
            "featureType": "POINT",
            "trackedEntityTypeAttributes": te_type_attributes
        }

        metadata = {"trackedEntityTypes": [tracked_entity_type]}
        response = dhis2_post("metadata", metadata)

        if response.status_code in [200, 201]:
            result = response.json()
            if result.get("status") == "OK":
                # Get the created type ID
                get_response = dhis2_get("trackedEntityTypes?filter=code:eq:WATER_FACILITY&fields=id,name,code")
                if get_response.status_code == 200:
                    type_data = get_response.json()
                    if type_data.get("trackedEntityTypes"):
                        te_type_id = type_data["trackedEntityTypes"][0]["id"]
                        print(f"✓ Created Tracked Entity Type: Water Facility (ID: {te_type_id})")
            else:
                print(f"✗ Failed: {result}")
        else:
            print(f"✗ Error: {response.status_code}")
            print(response.text)

✓ Tracked Entity Type already exists: Water Facility (ID: GNA3sINa6Hi)


## 5. Import Organisation Units

First, let's check existing org units and get the root org unit.

In [11]:
# Check existing organisation units
response = dhis2_get("organisationUnits?filter=level:eq:1&fields=id,name,code,level")

if response.status_code == 200:
    org_units = response.json()
    if org_units.get("organisationUnits"):
        root_ou = org_units["organisationUnits"][0]
        ROOT_OU_ID = root_ou["id"]
        print(f"Root Organisation Unit: {root_ou['name']} (ID: {ROOT_OU_ID})")
    else:
        print("No root organisation unit found. Will create Liberia as root.")
        ROOT_OU_ID = None
else:
    print(f"Error: {response.status_code}")
    ROOT_OU_ID = None

Root Organisation Unit: Liberia (ID: s2WkJ0c3GdX)


In [12]:
# Load organisation units from CSV
import os

csv_path = "org_units_sample.csv"

if os.path.exists(csv_path):
    org_df = pd.read_csv(csv_path)
    print(f"Loaded {len(org_df)} organisation units from {csv_path}")
    display(org_df)
else:
    print(f"CSV file not found: {csv_path}")
    print("Please create the CSV file first, then re-run this cell.")

Loaded 15 organisation units from org_units_sample.csv


,name,short_name,code,level,parent_code
0,Liberia,Liberia,LR,1,NaN
1,Montserrado County,Montserrado,LR_MON,2,LR
2,Nimba County,Nimba,LR_NIM,2,LR
3,Greater Monrovia District,Greater Monrovia,LR_MON_GM,3,LR_MON
4,St. Paul River District,St. Paul River,LR_MON_SPR,3,LR_MON
5,Sanniquellie-Mahn District,Sanniquellie-Mahn,LR_NIM_SM,3,LR_NIM
6,Saclepea-Mahn District,Saclepea-Mahn,LR_NIM_SAC,3,LR_NIM
7,Congo Town Clan,Congo Town,LR_MON_GM_CT,4,LR_MON_GM
8,Paynesville Clan,Paynesville,LR_MON_GM_PV,4,LR_MON_GM
9,Johnsonville Clan,Johnsonville,LR_MON_SPR_JV,4,LR_MON_SPR


In [13]:
# Create organisation units from CSV
# This creates them level by level (Country -> Counties -> Districts -> Clans)

created_org_units = {}

if 'org_df' in dir() and not org_df.empty:
    # Sort by level to create parents first
    org_df_sorted = org_df.sort_values('level')
    
    for _, row in org_df_sorted.iterrows():
        # Check if org unit already exists
        check_response = dhis2_get(f"organisationUnits?filter=code:eq:{row['code']}&fields=id,name,code")
        if check_response.status_code == 200:
            check_data = check_response.json()
            if check_data.get("organisationUnits"):
                # Already exists - reuse it
                existing_ou = check_data["organisationUnits"][0]
                created_org_units[row['code']] = existing_ou['id']
                level_indicator = "  " * (int(row['level']) - 1)
                print(f"{level_indicator}✓ {row['name']} (exists, ID: {existing_ou['id']})")
                continue
        
        # Determine parent
        if pd.isna(row.get('parent_code')) or row['parent_code'] == '':
            parent_id = None
        else:
            parent_id = created_org_units.get(row['parent_code'])
        
        # Build org unit payload
        org_unit = {
            "name": row['name'],
            "shortName": row['short_name'],
            "code": row['code'],
            "openingDate": "2020-01-01"
        }
        
        if parent_id:
            org_unit["parent"] = {"id": parent_id}
        
        # Create via metadata endpoint
        metadata = {"organisationUnits": [org_unit]}
        response = dhis2_post("metadata", metadata)
        
        if response.status_code in [200, 201]:
            result = response.json()
            if result.get("status") == "OK":
                # Get the created org unit ID
                get_response = dhis2_get(f"organisationUnits?filter=code:eq:{row['code']}&fields=id,name,code")
                if get_response.status_code == 200:
                    ou_data = get_response.json()
                    if ou_data.get("organisationUnits"):
                        ou_id = ou_data["organisationUnits"][0]["id"]
                        created_org_units[row['code']] = ou_id
                        level_indicator = "  " * (int(row['level']) - 1)
                        print(f"{level_indicator}✓ {row['name']} (created, ID: {ou_id})")
            else:
                print(f"✗ Failed: {row['name']} - {result}")
        else:
            print(f"✗ Error: {row['name']} - {response.status_code}")
    
    print(f"\nTotal organisation units ready: {len(created_org_units)}")
else:
    print("No organisation units to create. Load CSV first.")

✓ Liberia (exists, ID: s2WkJ0c3GdX)
  ✓ Montserrado County (exists, ID: lpWLxNCBuXQ)
  ✓ Nimba County (exists, ID: dJ18ADSMNCa)
    ✓ Greater Monrovia District (exists, ID: CA9Tm4fSBU5)
    ✓ St. Paul River District (exists, ID: Bud1Sv4A9qD)
    ✓ Sanniquellie-Mahn District (exists, ID: DTWXdzuwcHX)
    ✓ Saclepea-Mahn District (exists, ID: Sr1oAqp2UE9)
      ✓ Congo Town Clan (exists, ID: TEs9KJoNTxz)
      ✓ Paynesville Clan (exists, ID: hxczwhUgyG7)
      ✓ Johnsonville Clan (exists, ID: RqQgW18JgPp)
      ✓ Gardnersville Clan (exists, ID: o667do2tlP3)
      ✓ Sanniquellie Clan (exists, ID: toTAp3Qh0S4)
      ✓ Karnplay Clan (exists, ID: U1zeEnd5irf)
      ✓ Saclepea Clan (exists, ID: OZEoI8lKrua)
      ✓ Ganta Clan (exists, ID: niZqdGnIuxV)

Total organisation units ready: 15


In [14]:
# Assign admin user to organisation units (required for Tracker Capture)
if created_org_units:
    # Get root org unit (Liberia)
    root_ou_id = created_org_units.get("LR")
    
    if root_ou_id:
        # Get current user ID
        response = dhis2_get("me?fields=id,organisationUnits")
        if response.status_code == 200:
            user_data = response.json()
            user_id = user_data.get("id")
            current_org_units = user_data.get("organisationUnits", [])
            
            # Check if already assigned
            already_assigned = any(ou.get("id") == root_ou_id for ou in current_org_units)
            
            if already_assigned:
                print(f"✓ Admin user already assigned to Liberia org unit")
            else:
                # Use POST to collection endpoints (correct DHIS2 API method)
                success_count = 0
                
                # Assign to organisationUnits
                response = requests.post(
                    f"{BASE_URL}/users/{user_id}/organisationUnits/{root_ou_id}",
                    auth=AUTH,
                    headers=HEADERS
                )
                if response.status_code in [200, 201, 204]:
                    success_count += 1
                
                # Assign to dataViewOrganisationUnits
                response = requests.post(
                    f"{BASE_URL}/users/{user_id}/dataViewOrganisationUnits/{root_ou_id}",
                    auth=AUTH,
                    headers=HEADERS
                )
                if response.status_code in [200, 201, 204]:
                    success_count += 1
                
                # Assign to teiSearchOrganisationUnits
                response = requests.post(
                    f"{BASE_URL}/users/{user_id}/teiSearchOrganisationUnits/{root_ou_id}",
                    auth=AUTH,
                    headers=HEADERS
                )
                if response.status_code in [200, 201, 204]:
                    success_count += 1
                
                if success_count == 3:
                    print(f"✓ Assigned admin user to Liberia org unit")
                    print(f"  User can now access Tracker Capture")
                else:
                    print(f"⚠ Partial assignment ({success_count}/3 succeeded)")
else:
    print("No org units found. Run org unit creation cell first.")

✓ Admin user already assigned to Liberia org unit


## 6. Create Tracker Program

In [15]:
# Get all organisation units for program assignment
response = dhis2_get("organisationUnits?fields=id&paging=false")
all_org_units = []

if response.status_code == 200:
    ou_data = response.json()
    all_org_units = [{"id": ou["id"]} for ou in ou_data.get("organisationUnits", [])]
    print(f"Found {len(all_org_units)} organisation units for program assignment")

Found 15 organisation units for program assignment


In [16]:
# Check for existing program and create if needed
response = dhis2_get("programs?filter=code:eq:WF_REGISTRY&fields=id,name,code")
program_id = None

if response.status_code == 200:
    data = response.json()
    if data.get("programs"):
        program_id = data["programs"][0]["id"]
        print(f"✓ Program already exists: Water Facility Registry (ID: {program_id})")
    else:
        # Build program tracked entity attributes
        MANDATORY_ATTRS = ["GEO_CODE", "COUNTY", "DISTRICT", "COMMUNITY", "WATER_POINT_TYPE_ATTR"]

        program_attributes = []
        for i, (code, attr_info) in enumerate(created_attributes.items()):
            program_attributes.append({
                "trackedEntityAttribute": {"id": attr_info["id"]},
                "displayInList": code in ["SUNBIRD_OSID", "WF_ID", "SYNC_STATUS_ATTR", "GEO_CODE", "COUNTY", "DISTRICT", "WATER_POINT_TYPE_ATTR"],
                "mandatory": code in MANDATORY_ATTRS,
                "searchable": attr_info.get("searchable", False),
                "sortOrder": i + 1
            })

        # Create the tracker program
        program = {
            "name": "Water Facility Registry",
            "shortName": "WF Registry",
            "code": "WF_REGISTRY",
            "programType": "WITH_REGISTRATION",
            "trackedEntityType": {"id": te_type_id},
            "displayFrontPageList": True,
            "featureType": "POINT",
            "onlyEnrollOnce": True,
            "programTrackedEntityAttributes": program_attributes,
            "organisationUnits": all_org_units if all_org_units else []
        }

        metadata = {"programs": [program]}
        response = dhis2_post("metadata", metadata)

        if response.status_code in [200, 201]:
            result = response.json()
            if result.get("status") == "OK":
                # Get the created program ID
                get_response = dhis2_get("programs?filter=code:eq:WF_REGISTRY&fields=id,name,code")
                if get_response.status_code == 200:
                    prog_data = get_response.json()
                    if prog_data.get("programs"):
                        program_id = prog_data["programs"][0]["id"]
                        print(f"✓ Created Program: Water Facility Registry (ID: {program_id})")
            else:
                print(f"✗ Failed: {result}")
                print(json.dumps(result, indent=2))
        else:
            print(f"✗ Error: {response.status_code}")
            print(response.text)

✓ Program already exists: Water Facility Registry (ID: umMm1RGNPEG)


## 7. Verify Setup

In [17]:
# Summary of created resources
print("=" * 60)
print("DHIS2 Water Facility Setup Summary")
print("=" * 60)

# Option Sets
response = dhis2_get("optionSets?filter=code:in:[WATER_POINT_TYPE,EXTRACTION_TYPE,PUMP_TYPE,INSTALLER_TYPE,OWNER_TYPE,SYNC_STATUS]&fields=id,name,code")
if response.status_code == 200:
    data = response.json()
    print(f"\nOption Sets: {len(data.get('optionSets', []))}")
    for os in data.get('optionSets', []):
        print(f"  - {os['name']} ({os['code']})")

# Tracked Entity Attributes
response = dhis2_get("trackedEntityAttributes?paging=false&fields=id,name,code")
if response.status_code == 200:
    data = response.json()
    attrs = [a for a in data.get('trackedEntityAttributes', []) if a['code'] in created_attributes]
    print(f"\nTracked Entity Attributes: {len(attrs)}")

# Tracked Entity Type
response = dhis2_get("trackedEntityTypes?filter=code:eq:WATER_FACILITY&fields=id,name,code")
if response.status_code == 200:
    data = response.json()
    print(f"\nTracked Entity Type: {len(data.get('trackedEntityTypes', []))}")
    for tet in data.get('trackedEntityTypes', []):
        print(f"  - {tet['name']} ({tet['code']})")

# Program
response = dhis2_get("programs?filter=code:eq:WF_REGISTRY&fields=id,name,code,programType")
if response.status_code == 200:
    data = response.json()
    print(f"\nPrograms: {len(data.get('programs', []))}")
    for prog in data.get('programs', []):
        print(f"  - {prog['name']} ({prog['code']}) - {prog['programType']}")

# Organisation Units
response = dhis2_get("organisationUnits?paging=false&fields=id,name,level")
if response.status_code == 200:
    data = response.json()
    print(f"\nOrganisation Units: {len(data.get('organisationUnits', []))}")
    levels = {}
    for ou in data.get('organisationUnits', []):
        level = ou.get('level', 0)
        levels[level] = levels.get(level, 0) + 1
    for level in sorted(levels.keys()):
        print(f"  - Level {level}: {levels[level]} units")

print("\n" + "=" * 60)
print("Setup complete! You can now:")
print("1. Go to Tracker Capture app")
print("2. Select an organisation unit")
print("3. Register new Water Facilities")
print("=" * 60)

DHIS2 Water Facility Setup Summary

Option Sets: 6
  - Extraction Type (EXTRACTION_TYPE)
  - Installer Type (INSTALLER_TYPE)
  - Owner Type (OWNER_TYPE)
  - Pump Type (PUMP_TYPE)
  - Sync Status (SYNC_STATUS)
  - Water Point Type (WATER_POINT_TYPE)

Tracked Entity Attributes: 17

Tracked Entity Type: 1
  - Water Facility (WATER_FACILITY)

Programs: 1
  - Water Facility Registry (WF_REGISTRY) - WITH_REGISTRATION

Organisation Units: 15
  - Level 1: 1 units
  - Level 2: 2 units
  - Level 3: 4 units
  - Level 4: 8 units

Setup complete! You can now:
1. Go to Tracker Capture app
2. Select an organisation unit
3. Register new Water Facilities


## 8. Test: Create a Sample Water Facility

In [18]:
# Get a leaf organisation unit (lowest level) for testing
response = dhis2_get("organisationUnits?filter=level:ge:3&fields=id,name,code&pageSize=1")

test_ou_id = None
if response.status_code == 200:
    data = response.json()
    if data.get('organisationUnits'):
        test_ou = data['organisationUnits'][0]
        test_ou_id = test_ou['id']
        print(f"Test Organisation Unit: {test_ou['name']} (ID: {test_ou_id})")
    else:
        # Fallback to any org unit
        response = dhis2_get("organisationUnits?fields=id,name&pageSize=1")
        if response.status_code == 200:
            data = response.json()
            if data.get('organisationUnits'):
                test_ou = data['organisationUnits'][0]
                test_ou_id = test_ou['id']
                print(f"Test Organisation Unit: {test_ou['name']} (ID: {test_ou_id})")

Test Organisation Unit: Congo Town Clan (ID: TEs9KJoNTxz)


In [19]:
# Create a test water facility with enrollment (for verification)
import time
from datetime import datetime

if test_ou_id and te_type_id and program_id:
    # Use timestamp to generate unique GEO_CODE
    test_geo_code = f"TEST{int(time.time())}"
    today = datetime.now().strftime("%Y-%m-%d")
    
    # Build attributes array
    # Note: Option values must use the FULL option code (with prefix)
    test_attributes = [
        {"attribute": created_attributes["GEO_CODE"]["id"], "value": test_geo_code},
        {"attribute": created_attributes["COUNTY"]["id"], "value": "Montserrado"},
        {"attribute": created_attributes["DISTRICT"]["id"], "value": "Greater Monrovia"},
        {"attribute": created_attributes["COMMUNITY"]["id"], "value": "Congo Town"},
        {"attribute": created_attributes["WATER_POINT_TYPE_ATTR"]["id"], "value": "WATER_POINT_TYPE_TWB"},
        {"attribute": created_attributes["SYNC_STATUS_ATTR"]["id"], "value": "SYNC_STATUS_PENDING"},
        {"attribute": created_attributes["EXTRACTION_TYPE_ATTR"]["id"], "value": "EXTRACTION_TYPE_MANUAL"},
        {"attribute": created_attributes["INSTALLER"]["id"], "value": "INSTALLER_TYPE_NGO"},
        {"attribute": created_attributes["OWNER"]["id"], "value": "OWNER_TYPE_COMMUNITY"}
    ]
    
    # Create tracked entity instance WITH enrollment
    # This ensures the TEI shows up in Tracker Capture
    tei_payload = {
        "trackedEntityType": te_type_id,
        "orgUnit": test_ou_id,
        "attributes": test_attributes,
        "geometry": {
            "type": "Point",
            "coordinates": [-10.7957, 6.3156]  # Monrovia coordinates
        },
        "enrollments": [
            {
                "orgUnit": test_ou_id,
                "program": program_id,
                "enrollmentDate": today,
                "incidentDate": today
            }
        ]
    }
    
    response = dhis2_post("trackedEntityInstances", tei_payload)
    
    if response.status_code in [200, 201]:
        result = response.json()
        if result.get("status") == "OK" or result.get("response", {}).get("status") == "SUCCESS":
            tei_id = result.get("response", {}).get("importSummaries", [{}])[0].get("reference")
            print(f"✓ Created test Water Facility")
            print(f"  GEO_CODE: {test_geo_code}")
            print(f"  TEI ID: {tei_id}")
            print(f"  Enrolled in: Water Facility Registry")
            print(f"\nView in Tracker Capture:")
            print(f"  http://localhost:9090/dhis-web-tracker-capture/index.html")
        else:
            print(f"✗ Failed to create: {result}")
    else:
        print(f"✗ Error: {response.status_code}")
        print(response.text)
else:
    print("Cannot create test - missing organisation unit, tracked entity type, or program")

✓ Created test Water Facility
  GEO_CODE: TEST1777272225
  TEI ID: mo6iufj0rUI
  Enrolled in: Water Facility Registry

View in Tracker Capture:
  http://localhost:9090/dhis-web-tracker-capture/index.html
